# 02_train_baseline_model (NACC) — Updated

Trains a simple baseline 4-class classifier using `output/cohort.csv` produced by `01_build_cohort`.


Vascular Composite Engineering (E8)
Raw vascular risk variables were replaced with:
•	VASC_RISK_COUNT
•	VASC_MED_COUNT
•	STROKE_TIA_ANY
To reduce multicollinearity and improve balanced accuracy slightly (~+0.002–0.003).
Conclusion:
Aggregated vascular burden provided modest improvement but did not dramatically improve subtype separability.

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier


exp_name = "E8b_vascular_signal"
H = 5  # horizon years used in build_cohort notebook

OUT_DIR = os.path.join("..", "output")

cohort_file = f"cohort_{exp_name}_H{H}.csv"
DATA_PATH = os.path.join(OUT_DIR, cohort_file)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Could not find cohort file: {DATA_PATH}\n"
        f"Make sure 01_build_cohort exported: {cohort_file}"
    )

df = pd.read_csv(DATA_PATH)

print("Loaded:", cohort_file, df.shape)
print(df["label"].value_counts(dropna=False))
df.head()

Loaded: cohort_E8b_vascular_signal_H5.csv (38341, 69)
label
Censored         23300
NoDementia_Hy    11083
AD                3013
MixedOrOther       504
PD_LBD             289
Vascular           152
Name: count, dtype: int64


,NACCID,baseline_date,label,max_follow_years,has_H_followup,has_dementia_H,NACCAGE,SEX,RACE,HISPANIC,...,FOCLSIGN,HACHIN,HACHIN_GE_4,HACHIN_GE_7,STROKE_TIA_ANY,VASC_MED_COUNT,GAIT_ANY,NACCMOM,NACCDAD,SIVDFIND
0,NACC000011,2006-04-17,Censored,3.296372,False,False,62.0,2.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0,0.0,0,NaN,NaN,NaN
1,NACC000034,2015-07-16,NoDementia_Hy,6.986995,True,False,79.0,2.0,1.0,0.0,...,NaN,NaN,NaN,NaN,0,2.0,0,1.0,1.0,0.0
2,NACC000067,2012-05-23,Censored,2.088980,False,False,60.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0,1.0,1,1.0,0.0,NaN
3,NACC000073,2016-05-20,Censored,0.000000,False,False,44.0,2.0,1.0,0.0,...,NaN,NaN,NaN,NaN,0,0.0,0,0.0,0.0,0.0
4,NACC000138,2023-07-13,Censored,1.043121,False,False,54.0,2.0,50.0,1.0,...,NaN,NaN,NaN,NaN,0,0.0,0,0.0,0.0,0.0


In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

import matplotlib.pyplot as plt

FIG_DIR = os.path.join(OUT_DIR, "figs_baseline")
os.makedirs(FIG_DIR, exist_ok=True)

RANDOM_STATE = 42

## Keep only the 4-class task

In [3]:
def fix_categorical_codes(X):
    X = X.copy()
    CAT_CODE_VARS = [c for c in ["SEX", "RACE", "HISPANIC", "NACCUDSD"] if c in X.columns]
    STD_MISS = {-4, 9}

    for c in CAT_CODE_VARS:
        s = pd.to_numeric(X[c], errors="coerce")
        s = s.mask(s.isin(list(STD_MISS)), np.nan)
        X[c] = s

    if "RACE" in X.columns:
        X["RACE"] = X["RACE"].mask(X["RACE"].isin([99]), np.nan)

    for c in CAT_CODE_VARS:
        s = pd.to_numeric(X[c], errors="coerce")
        X[c] = s.map(lambda v: np.nan if pd.isna(v) else str(int(v))).astype(object)

    if "EDUC" in X.columns:
        educ_num = pd.to_numeric(X["EDUC"], errors="coerce")
        #X["EDUC_BIN"] = np.where(educ_num <= 12, "<=12", ">12").astype(object)

    X = X.replace({pd.NA: np.nan})
    for c in X.columns:
        if X[c].dtype == "bool":
            X[c] = X[c].astype(int)
    return X

In [4]:
import numpy as np
import pandas as pd


df4 = df.copy()  

# Keeping only non-censored outcome classes for training
KEEP = {"AD", "NoDementia_Hy", "PD_LBD", "Vascular"}
df4 = df4[df4["label"].isin(KEEP)].copy()

print("4-class rows:", len(df4))
print(df4["label"].value_counts())

# Dropping IDs/labels
#y = df4["label"].astype(str)
# drop_cols = [c for c in ["NACCID", "baseline_date", "label"] if c in df4.columns]
#drop_cols = [c for c in [ "NACCID", "baseline_date", "label", "has_H_followup", "has_dementia_H", "max_follow_years", "abs_days_from_baseline"] if c in df4.columns]


def prepare_experiment_df(df, exp_name):
    df = df.copy()

    # columns that are always dropped
    drop_cols = [c for c in ["NACCID","baseline_date","label","has_H_followup","has_dementia_H","max_follow_years","abs_days_from_baseline", "NACCUDSD"]
                 if c in df.columns]

    # E8: engineered vascular burden; dropping raw components to avoid collinearity
    if exp_name == "E8_clean_vascular":
        raw_vasc = ["DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","CVCHF","CVANGINA","CVBYPASS","CVHVALVE"]
        drop_cols += [c for c in raw_vasc if c in df.columns]

        raw_meds = ["NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD"]
        drop_cols += [c for c in raw_meds if c in df.columns]

        # drop_cols += [c for c in ["GAITDIS"] if c in df.columns]

    # E9: exec ratio; Dropping raw Trails
    if exp_name == "E9_clean_exec_ratio":
        drop_cols += [c for c in ["TRAILA","TRAILB","TRAILA_missing","TRAILB_missing"] if c in df.columns]

    # E10: building exec-memory axis (train-only z-scoring later), then Dropping raw
    # Here, what is dropped is only marked AFTER computing engineered features per split.

    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df["label"].astype(str)

    return X, y


from sklearn.model_selection import train_test_split

X, y = prepare_experiment_df(df4, exp_name)

# Applying categorical cleanup before splitting
X = fix_categorical_codes(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train label counts:\n", y_train.value_counts())
print("Test label counts:\n", y_test.value_counts())



def z_from_train(train_s, s):
    mu = train_s.mean()
    sd = train_s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return s*0.0
    return (s - mu) / sd

if exp_name == "E10_exec_mem_axis_raw":
    needed = ["NACCMMSE", "NACCMOCA", "TRAILB"]
    missing = [c for c in needed if c not in X_train.columns]
    if missing:
        raise ValueError(f"E10 missing required columns: {missing}")

    # Z-scores using train stats only
    X_train["Z_TRAILB"] = z_from_train(X_train["TRAILB"], X_train["TRAILB"])
    X_test["Z_TRAILB"]  = z_from_train(X_train["TRAILB"], X_test["TRAILB"])

    mem_cols = [c for c in ["NACCMMSE","NACCMOCA"] if c in X_train.columns]
    Zm_train = []
    Zm_test  = []
    for c in mem_cols:
        Zm_train.append(z_from_train(X_train[c], X_train[c]))
        Zm_test.append(z_from_train(X_train[c], X_test[c]))

    X_train["Z_MEM"] = np.nanmean(np.vstack([z.values for z in Zm_train]), axis=0)
    X_test["Z_MEM"]  = np.nanmean(np.vstack([z.values for z in Zm_test]), axis=0)

    # Executive minus memory axis
    X_train["EXEC_MINUS_MEM"] = X_train["Z_TRAILB"] - X_train["Z_MEM"]
    X_test["EXEC_MINUS_MEM"]  = X_test["Z_TRAILB"]  - X_test["Z_MEM"]

    # Dropping raw components to avoid collinearity
    drop_raw = [
        "TRAILA","TRAILB","NACCMMSE","NACCMOCA",
        "TRAILA_missing","TRAILB_missing","NACCMMSE_missing","NACCMOCA_missing",
        "Z_TRAILB","Z_MEM"
    ]
    X_train = X_train.drop(columns=[c for c in drop_raw if c in X_train.columns])
    X_test  = X_test.drop(columns=[c for c in drop_raw if c in X_test.columns])

    print("E10 engineered EXEC_MINUS_MEM added. Shapes now:", X_train.shape, X_test.shape)

labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

4-class rows: 14537
label
NoDementia_Hy    11083
AD                3013
PD_LBD             289
Vascular           152
Name: count, dtype: int64
Train: (11629, 62) Test: (2908, 62)
Train label counts:
 label
NoDementia_Hy    8866
AD               2410
PD_LBD            231
Vascular          122
Name: count, dtype: int64
Test label counts:
 label
NoDementia_Hy    2217
AD                603
PD_LBD             58
Vascular           30
Name: count, dtype: int64


In [5]:
csf_cols = ["CSFABETA", "CSFPTAU", "CSFTTAU"]
present_csf_cols = [c for c in csf_cols if c in df.columns]

if len(present_csf_cols) == len(csf_cols):
    df_csf = df[df[present_csf_cols].notna().all(axis=1)].copy()
    print("CSF subset size:", len(df_csf))
    print(df_csf["label"].value_counts())
else:
    print("CSF columns not present in this dataset:", 
          [c for c in csf_cols if c not in df.columns])

CSF columns not present in this dataset: ['CSFABETA', 'CSFPTAU', 'CSFTTAU']


## Split + preprocess + train

Train/test split

In [6]:
# X_train, X_test, y_train, y_test = train_test_split(
#     X_all, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
# )

Preprocessing pipelines (sparse for LR/boosting)

In [7]:
# cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object" or str(X_train[c].dtype).startswith("string")]
# num_cols = [c for c in X_train.columns if c not in cat_cols]

import pandas as pd

num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in X_train.columns if c not in num_cols]


numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

print("Num cols:", len(num_cols), "Cat cols:", len(cat_cols))
print(sorted([c for c in X_train.columns if "VASC" in c or "EXEC" in c or "TRAIL" in c or "MMSE" in c or "MOCA" in c]))
print(sorted([c for c in X_train.columns]))



Num cols: 59 Cat cols: 3
['NACCMMSE', 'NACCMMSE_missing', 'NACCMOCA', 'NACCMOCA_missing', 'TRAILA', 'TRAILA_missing', 'TRAILB', 'TRAILB_missing', 'VASC', 'VASC_MED_COUNT']
['ANIMALS', 'APNEA', 'BEREM', 'BEVHALL', 'BOSTON', 'BRADY', 'CBSTROKE', 'CBTIA', 'CDRGLOB', 'CDRSUM', 'CDRSUM_missing', 'COGFLUC', 'CVAFIB', 'CVANGINA', 'CVANGIO', 'CVBYPASS', 'CVCHF', 'CVD', 'CVDCOG', 'CVDIMAG', 'CVHVALVE', 'CVOTHR', 'DEP2YRS', 'DIABETES', 'EDUC', 'FOCLSIGN', 'FOCLSYM', 'GAITDIS', 'GAIT_ANY', 'HACHIN', 'HACHIN_GE_4', 'HACHIN_GE_7', 'HISPANIC', 'HYPERCHO', 'HYPERTEN', 'IMAGEWMH', 'IMAGMWMH', 'NACCAGE', 'NACCDAD', 'NACCMMSE', 'NACCMMSE_missing', 'NACCMOCA', 'NACCMOCA_missing', 'NACCMOM', 'PARKGAIT', 'PARKSIGN', 'RACE', 'RESTTRL', 'RESTTRR', 'RIGIDL', 'RIGIDR', 'SEX', 'SIVDFIND', 'SLOWINGL', 'SLOWINGR', 'STROKE_TIA_ANY', 'TRAILA', 'TRAILA_missing', 'TRAILB', 'TRAILB_missing', 'VASC', 'VASC_MED_COUNT']


In [8]:
# from imblearn.pipeline import Pipeline as ImbPipeline
# from imblearn.over_sampling import SMOTE

In [9]:
print(X_train)

       NACCAGE SEX RACE HISPANIC  EDUC  NACCMMSE  NACCMOCA  TRAILA  TRAILB  \
28423     68.0   1    1        0  19.0       NaN      22.0    61.0   197.0   
36733     79.0   2    1        0  12.0      26.0       NaN    66.0   155.0   
30320     71.0   2    2        0   8.0      28.0       NaN    34.0   186.0   
2618      74.0   1    1        0  12.0      30.0       NaN    31.0   132.0   
268       69.0   2    2        0  16.0      29.0       NaN    30.0   102.0   
...        ...  ..  ...      ...   ...       ...       ...     ...     ...   
25397     74.0   1    1        0  14.0      28.0       NaN    54.0   100.0   
9847      68.0   2   50        1  13.0      29.0       NaN    27.0    60.0   
21462     74.0   2    1        0  18.0      29.0       NaN    33.0   115.0   
36308     66.0   1    1        0  16.0       NaN      26.0    35.0    85.0   
5029      80.0   2   50        1   8.0      23.0       NaN   150.0   996.0   

       ANIMALS  ...  FOCLSIGN  HACHIN  HACHIN_GE_4  HACHIN_GE_7

Model A: Logistic Regression (final settings)

# Section 3C: Logistic Regression (balanced weights + vascular boost + threshold)

In [10]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.pipeline import Pipeline
# from sklearn.base import clone
# from sklearn.utils.class_weight import compute_class_weight
# from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
# import numpy as np

# # 1) Systematic class weights (balanced) + optional vascular boost
# classes = np.unique(y_train)

# base_w = compute_class_weight(
#     class_weight="balanced",
#     classes=classes,
#     y=y_train
# )
# class_weight_dict = dict(zip(classes, base_w))

# VAS_MULTIPLIER = 1.1   # try: 1.0, 1.2, 1.3, 1.5
# class_weight_dict["Vascular"] = class_weight_dict.get("Vascular", 1.0) * VAS_MULTIPLIER

# print("Final class weights:", class_weight_dict)

# # 2) Fit multinomial logistic regression
# clf_lr = Pipeline(steps=[
#     ("preprocess", clone(preprocess)),
#     ("model", LogisticRegression(
#         solver="lbfgs",          # supports multinomial automatically
#         max_iter=8000,
#         class_weight=class_weight_dict,
#         C=0.1
#     ))
# ])

# clf_lr.fit(X_train, y_train)

# # 3) Default prediction
# pred_lr = clf_lr.predict(X_test)

# # 4) OPTIONAL: vascular threshold override (often boosts vascular recall)
# USE_VASC_THRESHOLD = False
# T_VAS = 0.18  # try: 0.15–0.25

# if USE_VASC_THRESHOLD:
#     probs = clf_lr.predict_proba(X_test)
#     order = clf_lr.named_steps["model"].classes_
#     vas_idx = list(order).index("Vascular")
#     p_vas = probs[:, vas_idx]

#     pred_lr = pred_lr.copy()
#     pred_lr[p_vas >= T_VAS] = "Vascular"

# print("\n=== Logistic Regression (balanced weights + vascular boost + optional threshold) ===")
# print(classification_report(y_test, pred_lr, digits=4, labels=labels))
# print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
# print(confusion_matrix(y_test, pred_lr, labels=labels))

# Section 3A: Logistic Regression (custom weights, C=0.1)

In [11]:
from sklearn.linear_model import LogisticRegression

custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0
}

clf_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=8000,
        class_weight=custom_w,
        C=0.1
    ))
])

clf_lr.fit(X_train, y_train)
pred_lr = clf_lr.predict(X_test)

print("=== Logistic Regression (custom weights, C=0.1) ===")
print(classification_report(y_test, pred_lr, digits=4))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
print(confusion_matrix(y_test, pred_lr, labels=labels))


=== Logistic Regression (custom weights, C=0.1) ===
               precision    recall  f1-score   support

           AD     0.6235    0.7114    0.6646       603
NoDementia_Hy     0.9250    0.8791    0.9015      2217
       PD_LBD     0.4512    0.6379    0.5286        58
     Vascular     0.3226    0.3333    0.3279        30

     accuracy                         0.8339      2908
    macro avg     0.5806    0.6405    0.6056      2908
 weighted avg     0.8468    0.8339    0.8390      2908

Balanced accuracy: 0.6404557690758564
[[ 429  142   20   12]
 [ 235 1949   25    8]
 [  14    6   37    1]
 [  10   10    0   10]]


In [12]:
# ===== Inspect data after preprocessing / cleansing =====

import pandas as pd
import numpy as np

# Fit preprocess on training data (same as model will do)
X_train_clean = preprocess.fit_transform(X_train)
X_test_clean  = preprocess.transform(X_test)

print("Shape after preprocessing")
print("Train:", X_train_clean.shape)
print("Test :", X_test_clean.shape)

# Convert to DataFrame for easier inspection
try:
    feature_names = preprocess.get_feature_names_out()
except:
    feature_names = [f"f{i}" for i in range(X_train_clean.shape[1])]

df_clean = pd.DataFrame(X_train_clean, columns=feature_names)

print("\nFirst 10 rows of processed training data:")
display(df_clean.head(10))

print("\nSummary statistics:")
display(df_clean.describe())

print("\nCheck for NaNs:")
print(df_clean.isna().sum().sort_values(ascending=False).head(20))

print("\nCheck for infinite values:")
print(np.isinf(df_clean.values).sum())

print("\nUnique values in first few columns:")
for col in df_clean.columns[:10]:
    print(col, df_clean[col].unique()[:10])

Shape after preprocessing
Train: (11629, 69)
Test : (2908, 69)

First 10 rows of processed training data:


,num__NACCAGE,num__EDUC,num__NACCMMSE,num__NACCMOCA,num__TRAILA,num__TRAILB,num__ANIMALS,num__BOSTON,num__CDRSUM,num__CDRGLOB,...,cat__SEX_1,cat__SEX_2,cat__RACE_1,cat__RACE_2,cat__RACE_3,cat__RACE_4,cat__RACE_5,cat__RACE_50,cat__HISPANIC_0,cat__HISPANIC_1
0,-0.357331,0.490374,-0.078046,-1.497990,-0.052304,0.232158,-0.241323,-0.087303,3.281121,1.192036,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.780626,-0.716699,-0.406174,0.034348,-0.023784,0.029672,-0.464095,-0.478685,1.829875,1.192036,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.046979,-1.406455,-0.187422,0.034348,-0.206315,0.179126,-0.389838,-0.635238,-0.588868,-0.773674,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.263373,-0.716699,0.031330,0.034348,-0.223427,-0.081213,0.204223,-0.009026,-0.588868,-0.773674,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,-0.253881,-0.026943,-0.078046,0.034348,-0.229131,-0.225846,-0.835383,-0.009026,-0.588868,-0.773674,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
5,-2.426345,0.317935,0.031330,0.034348,-0.229131,-0.491007,0.055707,-0.087303,0.378629,1.192036,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
6,-0.357331,0.662813,-0.078046,0.034348,-0.080825,-0.351195,0.352738,-0.009026,-0.588868,-0.773674,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
7,-0.357331,-0.716699,0.031330,0.034348,-0.166386,-0.235489,-0.612610,-0.165579,-0.588868,-0.773674,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8,0.573725,-0.026943,-0.078046,0.034348,-0.280468,-0.384943,-0.612610,-0.009026,-0.588868,-0.773674,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
9,2.228936,0.662813,-0.078046,0.034348,5.275306,4.079397,0.055707,-0.322132,-0.105119,1.192036,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0



Summary statistics:


,num__NACCAGE,num__EDUC,num__NACCMMSE,num__NACCMOCA,num__TRAILA,num__TRAILB,num__ANIMALS,num__BOSTON,num__CDRSUM,num__CDRGLOB,...,cat__SEX_1,cat__SEX_2,cat__RACE_1,cat__RACE_2,cat__RACE_3,cat__RACE_4,cat__RACE_5,cat__RACE_50,cat__HISPANIC_0,cat__HISPANIC_1
count,1.162900e+04,1.162900e+04,1.162900e+04,1.162900e+04,1.162900e+04,1.162900e+04,1.162900e+04,1.162900e+04,1.162900e+04,1.162900e+04,...,11629.000000,11629.000000,11629.000000,11629.000000,11629.000000,11629.000000,11629.000000,11629.000000,11629.00000,11629.00000
mean,-6.928846e-16,1.222019e-17,-1.338110e-16,-1.075376e-16,8.554130e-18,-6.048992e-17,-8.645782e-17,-1.564184e-16,5.865689e-17,-3.421652e-17,...,0.402098,0.597902,0.818987,0.139909,0.005761,0.000430,0.026915,0.007997,0.94058,0.05942
std,1.000043e+00,1.000043e+00,1.000043e+00,1.000043e+00,1.000043e+00,1.000043e+00,1.000043e+00,1.000043e+00,1.000043e+00,1.000043e+00,...,0.490343,0.490343,0.385045,0.346907,0.075689,0.020732,0.161843,0.089073,0.23642,0.23642
min,-5.529866e+00,-2.785967e+00,-1.718685e+00,-1.018124e+01,-3.489166e-01,-6.356400e-01,-1.429443e+00,-2.279044e+00,-5.888680e-01,-7.736742e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000
25%,-5.642328e-01,-3.718212e-01,-1.874223e-01,3.434770e-02,-2.519473e-01,-4.138693e-01,-4.640951e-01,-1.655794e-01,-5.888680e-01,-7.736742e-01,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.00000,0.00000
50%,5.647132e-02,-2.694313e-02,-7.804639e-02,3.434770e-02,-2.120187e-01,-3.078051e-01,-1.670650e-01,-8.730294e-02,-5.888680e-01,-7.736742e-01,...,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.00000,0.00000
75%,6.771754e-01,3.179349e-01,3.132954e-02,3.434770e-02,-1.435698e-01,-9.567663e-02,2.042225e-01,-9.026465e-03,3.786293e-01,1.192036e+00,...,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.00000,0.00000
max,3.884147e+00,1.428550e+01,7.468893e+00,2.588244e+00,5.292418e+00,4.093861e+00,5.699278e+00,5.392051e+00,1.682608e+01,1.102059e+01,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.00000



Check for NaNs:
num__NACCAGE             0
num__EDUC                0
num__NACCMMSE            0
num__NACCMOCA            0
num__TRAILA              0
num__TRAILB              0
num__ANIMALS             0
num__BOSTON              0
num__CDRSUM              0
num__CDRGLOB             0
num__NACCMMSE_missing    0
num__NACCMOCA_missing    0
num__TRAILA_missing      0
num__TRAILB_missing      0
num__CDRSUM_missing      0
num__COGFLUC             0
num__BEVHALL             0
num__BEREM               0
num__PARKSIGN            0
num__BRADY               0
dtype: int64

Check for infinite values:
0

Unique values in first few columns:
num__NACCAGE [-0.35733143  0.78062613 -0.04697937  0.26337269 -0.25388074 -2.42634519
  0.57372476  2.22893576 -2.21944381  0.15992201]
num__EDUC [ 0.49037393 -0.71669922 -1.40645531 -0.02694313  0.31793491  0.66281295
 -0.37182118 -0.5442602   1.007691   -1.06157727]
num__NACCMMSE [-0.07804639 -0.40617419 -0.18742233  0.03132954 -0.51555013 -0.62492606
 -0.296

In [13]:
print("Training class distribution:")
print(y_train.value_counts(normalize=True))

Training class distribution:
label
NoDementia_Hy    0.762404
AD               0.207241
PD_LBD           0.019864
Vascular         0.010491
Name: proportion, dtype: float64


In [14]:
for col in ["RESTTRL","RESTTRR","PARKGAIT","BRADY"]:
    print(col)
    print(X_train[col].value_counts().head(20))

RESTTRL
RESTTRL
0.0    3221
1.0      41
Name: count, dtype: int64
RESTTRR
RESTTRR
0.0    3215
1.0      46
Name: count, dtype: int64
PARKGAIT
PARKGAIT
0.0    3183
1.0      80
Name: count, dtype: int64
BRADY
BRADY
0.0    3135
1.0     130
Name: count, dtype: int64


In [15]:
print("Max values:")
print(df_clean.max().sort_values(ascending=False).head())

print("Min values:")
print(df_clean.min().sort_values().head())

Max values:
num__SIVDFIND    26.135394
num__CVHVALVE    23.511324
num__IMAGEWMH    21.989581
num__CDRSUM      16.826082
num__RESTTRL     16.811726
dtype: float64
Min values:
num__NACCMOCA   -10.181237
num__NACCAGE     -5.529866
num__EDUC        -2.785967
num__BOSTON      -2.279044
num__NACCMMSE    -1.718685
dtype: float64


In [16]:
print("Missing values before preprocessing:")
print(X_train.isna().sum().sort_values(ascending=False).head(20))

Missing values before preprocessing:
CVDIMAG     10140
IMAGEWMH    10139
IMAGMWMH    10130
CVDCOG       9411
VASC         8579
APNEA        8516
NACCMOCA     8508
CVHVALVE     8406
CVANGINA     8401
RESTTRR      8368
RESTTRL      8367
PARKGAIT     8366
RIGIDL       8366
RIGIDR       8364
SLOWINGR     8364
BRADY        8364
SLOWINGL     8363
PARKSIGN     8357
SIVDFIND     8357
CVD          8348
dtype: int64


# Section 3G: Tuned Logistic Regression

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import numpy as np

custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0
}

# 1) Created train/validation split from TRAIN only
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# 2) Fitted LR on inner-train

clf_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=8000,
        class_weight=custom_w,
        C=0.1
    ))
])

clf_lr.fit(X_tr, y_tr)

# 3) Tuned vascular threshold on validation set
proba_val = clf_lr.predict_proba(X_val)
class_order = clf_lr.named_steps["model"].classes_
vas_idx = list(class_order).index("Vascular")

thresholds = np.linspace(0.05, 0.50, 20)
results = []

for T in thresholds:
    preds = []
    for p in proba_val:
        if p[vas_idx] >= T:
            preds.append("Vascular")
        else:
            preds.append(class_order[np.argmax(p)])
    ba = balanced_accuracy_score(y_val, preds)

    # vascular recall
    yv = np.array(y_val)
    pv = np.array(preds)
    denom = np.sum(yv == "Vascular")
    vrec = (np.sum((yv == "Vascular") & (pv == "Vascular")) / denom) if denom else 0.0

    results.append((T, ba, vrec))

best_T, best_ba, best_vrec = max(results, key=lambda x: x[1])
print(f"Best T_VAS (by val BA): {best_T:.3f} | val BA={best_ba:.4f} | val VascRecall={best_vrec:.4f}")

# 4) Retrained LR on FULL TRAIN and evaluate once on TEST using tuned threshold
clf_lr.fit(X_train, y_train)

class_order = clf_lr.named_steps["model"].classes_
vas_idx = list(class_order).index("Vascular")

proba_test = clf_lr.predict_proba(X_test)
preds_test = []

for p in proba_test:
    if p[vas_idx] >= best_T:
        preds_test.append("Vascular")
    else:
        preds_test.append(class_order[np.argmax(p)])

print("\n=== Logistic Regression (Vascular threshold tuned) ===")
print(classification_report(y_test, preds_test, digits=4, labels=labels))
print("Balanced accuracy:", balanced_accuracy_score(y_test, preds_test))
print(confusion_matrix(y_test, preds_test, labels=labels))

Best T_VAS (by val BA): 0.074 | val BA=0.6517 | val VascRecall=0.6000

=== Logistic Regression (Vascular threshold tuned) ===
               precision    recall  f1-score   support

           AD     0.6358    0.5473    0.5882       603
NoDementia_Hy     0.9331    0.8489    0.8890      2217
       PD_LBD     0.4861    0.6034    0.5385        58
     Vascular     0.0700    0.7000    0.1273        30

     accuracy                         0.7799      2908
    macro avg     0.5313    0.6749    0.5357      2908
 weighted avg     0.8536    0.7799    0.8118      2908

Balanced accuracy: 0.6749017151190526
[[ 330  124   17  132]
 [ 173 1882   20  142]
 [  13    5   35    5]
 [   3    6    0   21]]


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support
)
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

# --------------------------------------------------
# 0) Labels / weights
# --------------------------------------------------
labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

# Slightly reduced vascular weight to help precision/F1
custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 5.0,
    "Vascular": 6.0
}

# --------------------------------------------------
# 1) Train / validation split from TRAIN only
# --------------------------------------------------
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# --------------------------------------------------
# 2) Candidate model settings
#    One model change included: Elastic Net logistic regression
# --------------------------------------------------
model_settings = [
    {
        "name": "LR_L2_C0.1",
        "penalty": "l2",
        "solver": "lbfgs",
        "C": 0.1,
        "l1_ratio": None,
    },
    {
        "name": "LR_L2_C0.05",
        "penalty": "l2",
        "solver": "lbfgs",
        "C": 0.05,
        "l1_ratio": None,
    },
    {
        "name": "LR_ElasticNet_C0.1_l1r0.2",
        "penalty": "elasticnet",
        "solver": "saga",
        "C": 0.1,
        "l1_ratio": 0.2,
    },
    {
        "name": "LR_ElasticNet_C0.05_l1r0.2",
        "penalty": "elasticnet",
        "solver": "saga",
        "C": 0.05,
        "l1_ratio": 0.2,
    },
]

thresholds = np.linspace(0.05, 0.40, 36)

# --------------------------------------------------
# Helper: apply vascular threshold override
# --------------------------------------------------
def predict_with_vascular_threshold(proba, class_order, threshold):
    vas_idx = list(class_order).index("Vascular")
    preds = []
    for p in proba:
        if p[vas_idx] >= threshold:
            preds.append("Vascular")
        else:
            preds.append(class_order[np.argmax(p)])
    return np.array(preds)

# --------------------------------------------------
# 3) Validation search:
#    choose model + threshold by vascular F1
#    tie-breakers: macro F1, balanced accuracy
# --------------------------------------------------
search_rows = []

for cfg in model_settings:
    clf = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(
            max_iter=12000,
            class_weight=custom_w,
            C=cfg["C"],
            penalty=cfg["penalty"],
            solver=cfg["solver"],
            l1_ratio=cfg["l1_ratio"],
            n_jobs=None
        ))
    ])

    clf.fit(X_tr, y_tr)

    proba_val = clf.predict_proba(X_val)
    class_order = clf.named_steps["model"].classes_

    for T in thresholds:
        preds_val = predict_with_vascular_threshold(proba_val, class_order, T)

        # overall metrics
        ba = balanced_accuracy_score(y_val, preds_val)
        macro_f1 = f1_score(y_val, preds_val, average="macro")

        # per-class metrics
        p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
            y_val, preds_val, labels=labels, zero_division=0
        )

        metric_map = {
            lab: {"precision": p, "recall": r, "f1": f, "support": s}
            for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
        }

        vprec = metric_map["Vascular"]["precision"]
        vrec  = metric_map["Vascular"]["recall"]
        vf1   = metric_map["Vascular"]["f1"]

        search_rows.append({
            "model_name": cfg["name"],
            "penalty": cfg["penalty"],
            "solver": cfg["solver"],
            "C": cfg["C"],
            "l1_ratio": cfg["l1_ratio"],
            "threshold": T,
            "val_bal_acc": ba,
            "val_macro_f1": macro_f1,
            "val_vasc_precision": vprec,
            "val_vasc_recall": vrec,
            "val_vasc_f1": vf1,
        })

search_df = pd.DataFrame(search_rows)

# Sort by vascular F1 first, then macro F1, then BA
search_df = search_df.sort_values(
    ["val_vasc_f1", "val_macro_f1", "val_bal_acc"],
    ascending=[False, False, False]
).reset_index(drop=True)

print("Top validation settings:")
display(search_df.head(10))

best_row = search_df.iloc[0]
best_model_name = best_row["model_name"]
best_T = float(best_row["threshold"])

print(
    f"Best setting: {best_model_name} | "
    f"T_VAS={best_T:.3f} | "
    f"val VascF1={best_row['val_vasc_f1']:.4f} | "
    f"val MacroF1={best_row['val_macro_f1']:.4f} | "
    f"val BA={best_row['val_bal_acc']:.4f} | "
    f"val VascPrec={best_row['val_vasc_precision']:.4f} | "
    f"val VascRec={best_row['val_vasc_recall']:.4f}"
)

# --------------------------------------------------
# 4) Refit best model on FULL TRAIN
# --------------------------------------------------
best_cfg = next(cfg for cfg in model_settings if cfg["name"] == best_model_name)

clf_best = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=12000,
        class_weight=custom_w,
        C=best_cfg["C"],
        penalty=best_cfg["penalty"],
        solver=best_cfg["solver"],
        l1_ratio=best_cfg["l1_ratio"],
        n_jobs=None
    ))
])

clf_best.fit(X_train, y_train)

class_order = clf_best.named_steps["model"].classes_
proba_test = clf_best.predict_proba(X_test)
preds_test = predict_with_vascular_threshold(proba_test, class_order, best_T)

# --------------------------------------------------
# 5) Final test evaluation
# --------------------------------------------------
print("\n=== Logistic Regression (best model + vascular threshold tuned by Vascular F1) ===")
print(classification_report(y_test, preds_test, digits=4, labels=labels, zero_division=0))

test_bal_acc = balanced_accuracy_score(y_test, preds_test)
test_macro_f1 = f1_score(y_test, preds_test, average="macro")
test_weighted_f1 = f1_score(y_test, preds_test, average="weighted")
test_cm = confusion_matrix(y_test, preds_test, labels=labels)

print("Balanced accuracy:", test_bal_acc)
print("Macro F1:", test_macro_f1)
print("Weighted F1:", test_weighted_f1)
print(confusion_matrix(y_test, preds_test, labels=labels))

# Optional: print vascular-specific test metrics cleanly
p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
    y_test, preds_test, labels=labels, zero_division=0
)
test_metric_map = {
    lab: {"precision": p, "recall": r, "f1": f, "support": s}
    for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
}

print("\nTest vascular metrics:")
print(test_metric_map["Vascular"])

c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l2 with l1_ratio=None. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packag

Top validation settings:


,model_name,penalty,solver,C,l1_ratio,threshold,val_bal_acc,val_macro_f1,val_vasc_precision,val_vasc_recall,val_vasc_f1
0,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.36,0.643096,0.618481,0.214286,0.36,0.268657
1,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.35,0.643096,0.617663,0.209302,0.36,0.264706
2,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.34,0.642578,0.616375,0.204545,0.36,0.260870
3,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.27,0.650553,0.603857,0.192308,0.40,0.259740
4,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.28,0.650553,0.603857,0.192308,0.40,0.259740
5,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.31,0.642578,0.615612,0.200000,0.36,0.257143
6,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.32,0.642578,0.615612,0.200000,0.36,0.257143
7,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.33,0.642578,0.615612,0.200000,0.36,0.257143
8,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.26,0.649894,0.601802,0.185185,0.40,0.253165
9,LR_L2_C0.1,l2,lbfgs,0.10,NaN,0.26,0.635587,0.604633,0.191489,0.36,0.250000


Best setting: LR_ElasticNet_C0.1_l1r0.2 | T_VAS=0.360 | val VascF1=0.2687 | val MacroF1=0.6185 | val BA=0.6431 | val VascPrec=0.2143 | val VascRec=0.3600


c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(



=== Logistic Regression (best model + vascular threshold tuned by Vascular F1) ===
               precision    recall  f1-score   support

           AD     0.6234    0.7081    0.6630       603
NoDementia_Hy     0.9255    0.8800    0.9022      2217
       PD_LBD     0.4800    0.6207    0.5414        58
     Vascular     0.2500    0.3333    0.2857        30

     accuracy                         0.8336      2908
    macro avg     0.5697    0.6355    0.5981      2908
 weighted avg     0.8470    0.8336    0.8390      2908

Balanced accuracy: 0.6355417668474079
Macro F1: 0.5980769198064273
Weighted F1: 0.8390491242042618
[[ 427  141   18   17]
 [ 234 1951   21   11]
 [  14    6   36    2]
 [  10   10    0   10]]

Test vascular metrics:
{'precision': np.float64(0.25), 'recall': np.float64(0.3333333333333333), 'f1': np.float64(0.2857142857142857), 'support': np.int64(30)}


c:\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support
)
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

# --------------------------------------------------
# 0) Labels / weights
# --------------------------------------------------
labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 5.0,
    "Vascular": 6.0
}

# --------------------------------------------------
# 1) Candidate model settings
# --------------------------------------------------
model_settings = [
    {
        "name": "LR_L2_C0.1",
        "penalty": "l2",
        "solver": "lbfgs",
        "C": 0.1,
        "l1_ratio": None,
    },
    {
        "name": "LR_L2_C0.05",
        "penalty": "l2",
        "solver": "lbfgs",
        "C": 0.05,
        "l1_ratio": None,
    },
    {
        "name": "LR_ElasticNet_C0.1_l1r0.2",
        "penalty": "elasticnet",
        "solver": "saga",
        "C": 0.1,
        "l1_ratio": 0.2,
    },
    {
        "name": "LR_ElasticNet_C0.05_l1r0.2",
        "penalty": "elasticnet",
        "solver": "saga",
        "C": 0.05,
        "l1_ratio": 0.2,
    },
]

thresholds = np.linspace(0.05, 0.40, 36)

# --------------------------------------------------
# Helper: apply vascular threshold override
# --------------------------------------------------
def predict_with_vascular_threshold(proba, class_order, threshold):
    vas_idx = list(class_order).index("Vascular")
    preds = []
    for p in proba:
        if p[vas_idx] >= threshold:
            preds.append("Vascular")
        else:
            preds.append(class_order[np.argmax(p)])
    return np.array(preds)

# --------------------------------------------------
# 2) 3-fold CV search (FAST VERSION)
# --------------------------------------------------
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

cv_rows = []

for cfg in model_settings:
    print(f"Running model config: {cfg['name']}")
    
    # Store metrics by threshold across folds
    threshold_metric_store = {
        T: {
            "bal_acc": [],
            "macro_f1": [],
            "vasc_precision": [],
            "vasc_recall": [],
            "vasc_f1": []
        }
        for T in thresholds
    }

    for fold_id, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), start=1):
        print(f"  Fold {fold_id}/3")

        X_tr_fold = X_train.iloc[tr_idx]
        X_val_fold = X_train.iloc[val_idx]
        y_tr_fold = y_train.iloc[tr_idx]
        y_val_fold = y_train.iloc[val_idx]

        clf = Pipeline(steps=[
            ("preprocess", preprocess),
            ("model", LogisticRegression(
                max_iter=12000,
                class_weight=custom_w,
                C=cfg["C"],
                penalty=cfg["penalty"],
                solver=cfg["solver"],
                l1_ratio=cfg["l1_ratio"],
                n_jobs=None
            ))
        ])

        # FIT ONCE PER FOLD
        clf.fit(X_tr_fold, y_tr_fold)

        # PREDICT PROBA ONCE PER FOLD
        proba_val = clf.predict_proba(X_val_fold)
        class_order = clf.named_steps["model"].classes_

        # LOOP OVER THRESHOLDS USING SAME PROBAS
        for T in thresholds:
            preds_val = predict_with_vascular_threshold(proba_val, class_order, T)

            ba = balanced_accuracy_score(y_val_fold, preds_val)
            macro_f1 = f1_score(y_val_fold, preds_val, average="macro")

            p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
                y_val_fold, preds_val, labels=labels, zero_division=0
            )

            metric_map = {
                lab: {"precision": p, "recall": r, "f1": f, "support": s}
                for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
            }

            threshold_metric_store[T]["bal_acc"].append(ba)
            threshold_metric_store[T]["macro_f1"].append(macro_f1)
            threshold_metric_store[T]["vasc_precision"].append(metric_map["Vascular"]["precision"])
            threshold_metric_store[T]["vasc_recall"].append(metric_map["Vascular"]["recall"])
            threshold_metric_store[T]["vasc_f1"].append(metric_map["Vascular"]["f1"])

    # Aggregate across folds for this model config
    for T in thresholds:
        cv_rows.append({
            "model_name": cfg["name"],
            "penalty": cfg["penalty"],
            "solver": cfg["solver"],
            "C": cfg["C"],
            "l1_ratio": cfg["l1_ratio"],
            "threshold": T,
            "cv_bal_acc_mean": np.mean(threshold_metric_store[T]["bal_acc"]),
            "cv_bal_acc_std": np.std(threshold_metric_store[T]["bal_acc"]),
            "cv_macro_f1_mean": np.mean(threshold_metric_store[T]["macro_f1"]),
            "cv_macro_f1_std": np.std(threshold_metric_store[T]["macro_f1"]),
            "cv_vasc_precision_mean": np.mean(threshold_metric_store[T]["vasc_precision"]),
            "cv_vasc_recall_mean": np.mean(threshold_metric_store[T]["vasc_recall"]),
            "cv_vasc_f1_mean": np.mean(threshold_metric_store[T]["vasc_f1"]),
            "cv_vasc_f1_std": np.std(threshold_metric_store[T]["vasc_f1"]),
        })

cv_df = pd.DataFrame(cv_rows)

# Sort by mean vascular F1 first, then mean macro F1, then mean BA
cv_df = cv_df.sort_values(
    ["cv_vasc_f1_mean", "cv_macro_f1_mean", "cv_bal_acc_mean"],
    ascending=[False, False, False]
).reset_index(drop=True)

print("\nTop 15 CV settings:")
display(cv_df.head(15))

best_row = cv_df.iloc[0]
best_model_name = best_row["model_name"]
best_T = float(best_row["threshold"])

print(
    f"Best CV setting: {best_model_name} | "
    f"T_VAS={best_T:.3f} | "
    f"CV mean VascF1={best_row['cv_vasc_f1_mean']:.4f} ± {best_row['cv_vasc_f1_std']:.4f} | "
    f"CV mean MacroF1={best_row['cv_macro_f1_mean']:.4f} ± {best_row['cv_macro_f1_std']:.4f} | "
    f"CV mean BA={best_row['cv_bal_acc_mean']:.4f} ± {best_row['cv_bal_acc_std']:.4f}"
)

# --------------------------------------------------
# 3) Refit best model on FULL TRAIN
# --------------------------------------------------
best_cfg = next(cfg for cfg in model_settings if cfg["name"] == best_model_name)

clf_best = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=12000,
        class_weight=custom_w,
        C=best_cfg["C"],
        penalty=best_cfg["penalty"],
        solver=best_cfg["solver"],
        l1_ratio=best_cfg["l1_ratio"],
        n_jobs=None
    ))
])

clf_best.fit(X_train, y_train)

class_order = clf_best.named_steps["model"].classes_
proba_test = clf_best.predict_proba(X_test)
preds_test = predict_with_vascular_threshold(proba_test, class_order, best_T)

# --------------------------------------------------
# 4) Final test evaluation
# --------------------------------------------------
print("\n=== Logistic Regression (3-fold CV selected model + vascular threshold) ===")
print(classification_report(y_test, preds_test, digits=4, labels=labels, zero_division=0))

test_bal_acc = balanced_accuracy_score(y_test, preds_test)
test_macro_f1 = f1_score(y_test, preds_test, average="macro")
test_weighted_f1 = f1_score(y_test, preds_test, average="weighted")

print("Balanced accuracy:", test_bal_acc)
print("Macro F1:", test_macro_f1)
print("Weighted F1:", test_weighted_f1)
print(confusion_matrix(y_test, preds_test, labels=labels))

p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
    y_test, preds_test, labels=labels, zero_division=0
)
test_metric_map = {
    lab: {"precision": p, "recall": r, "f1": f, "support": s}
    for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
}

print("\nTest vascular metrics:")
print(test_metric_map["Vascular"])

Running model config: LR_L2_C0.1
  Fold 1/3


c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l2 with l1_ratio=None. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


  Fold 2/3


c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l2 with l1_ratio=None. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


  Fold 3/3


...

c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l2 with l1_ratio=None. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


  Fold 3/3


c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l2 with l1_ratio=None. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Running model config: LR_ElasticNet_C0.1_l1r0.2
  Fold 1/3


c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


# Section 3E: HistGradientBoosting (tuned + weighted)
Balanced accuracy = 0.613
Vascular recall = 0.176
Worse than logistic regression.


In [109]:
from sklearn.ensemble import HistGradientBoostingClassifier
import numpy as np

w_train = y_train.map({
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 10.0,
    "Vascular": 20.0
}).astype(float).values

clf_hgb = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=0.03,
        max_depth=8,
        max_iter=1200,
        min_samples_leaf=10,
        l2_regularization=0.0,
        random_state=42
    ))
])

clf_hgb.fit(X_train, y_train, model__sample_weight=w_train)
pred_hgb = clf_hgb.predict(X_test)

print("\n=== HistGradientBoosting tuned (weighted) ===")
print(classification_report(y_test, pred_hgb, digits=4))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_hgb))
print(confusion_matrix(y_test, pred_hgb, labels=labels))


=== HistGradientBoosting tuned (weighted) ===
               precision    recall  f1-score   support

           AD     0.6676    0.7622    0.7118       614
NoDementia_Hy     0.9384    0.8868    0.9119      2217
       PD_LBD     0.5068    0.6271    0.5606        59
     Vascular     0.1091    0.1765    0.1348        34

     accuracy                         0.8471      2924
    macro avg     0.5555    0.6131    0.5798      2924
 weighted avg     0.8632    0.8471    0.8537      2924

Balanced accuracy: 0.6131470395701917
[[ 468  112   16   18]
 [ 202 1966   19   30]
 [  14    7   37    1]
 [  17   10    1    6]]


# Section 3F: XGBoost (tuned + early stopping + strong class weighting)
Balanced accuracy = 0.569
Vascular recall = 0.029
Tree ensembles largely ignored vascular class despite weighting.


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]
label_to_int = {lab:i for i, lab in enumerate(labels)}
int_to_label = {i:lab for lab, i in label_to_int.items()}

y_train_int = y_train.map(label_to_int).astype(int)
y_test_int  = y_test.map(label_to_int).astype(int)

# stronger weights for tree models
TREE_WEIGHTS = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 10.0,
    "Vascular": 20.0
}
w_train = y_train.map(TREE_WEIGHTS).astype(float).values

# train/val split
X_tr, X_val, y_tr_int, y_val_int, w_tr, w_val = train_test_split(
    X_train, y_train_int, w_train,
    test_size=0.2, random_state=42, stratify=y_train_int
)

# preprocess
Xtr_mat = preprocess.fit_transform(X_tr)
Xval_mat = preprocess.transform(X_val)
Xte_mat  = preprocess.transform(X_test)

xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=len(labels),

    n_estimators=5000,
    learning_rate=0.02,
    max_depth=4,
    min_child_weight=1,
    subsample=0.85,
    colsample_bytree=0.85,

    reg_lambda=1.0,
    gamma=0.0,

    max_delta_step=1,

    tree_method="hist",
    random_state=42,
    eval_metric="mlogloss",

    early_stopping_rounds=200 
)

xgb.fit(
    Xtr_mat,
    y_tr_int,
    sample_weight=w_tr,
    eval_set=[(Xval_mat, y_val_int)],
    verbose=False
)

pred_int = xgb.predict(Xte_mat)
pred_xgb = pd.Series(pred_int).map(int_to_label)

print("\n=== XGBoost tuned (weighted + early stopping) ===")
print(classification_report(y_test, pred_xgb, digits=4))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_xgb))
print(confusion_matrix(y_test, pred_xgb, labels=labels))

NameError: name 'y_train' is not defined